In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/00.bronze-helper

In [0]:
from pyspark.sql import functions as F

In [0]:
source_file = f"{loading_folder_path}/{v_batch_id}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"
source_file1 = f"{loading_folder_path}/{v_batch_id}/races.csv"
table_name1 = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
df_circuits = (
    spark.read
         .format('csv')
         .option('header', 'true')
         .option('mode', 'failfast')
         .schema(circuits_schema)
         .load(source_file))

adding metadata column

In [0]:
final_df_circuits = add_ingestion_metadata(df_circuits)

write data to detla table in bronze layer


In [0]:
write_to_bronze(input_df=final_df_circuits, table_name=table_name, batch_id=v_batch_id)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
])

In [0]:
df_races = (
    spark.read
         .format('csv')
         .option('header', 'true')
         .option('mode', 'failfast')
         .schema(races_schema)
         .load(source_file1))

In [0]:
final_df_races = add_ingestion_metadata(df_races)

In [0]:
write_to_bronze(input_df=final_df_races, table_name=table_name1, batch_id=v_batch_id)